# CSE445 Machine Learning — Final Report
## Hospital-Onset Infection Prediction
### Part 1: Tabular → Text Conversion → Preprocessing → ML Models

**Pipeline:**
`Tabular CSV` → `Clinical Text Narratives` → `TF-IDF Features` → `Preprocessing` → `ML Models`

**Steps in this notebook:**
1. Convert tabular data to clinical text narratives
2. All preprocessing techniques on text features (TF-IDF)
3. ML models WITHOUT hyperparameter tuning (9 models + Stacking MUST)
4. ML models WITH hyperparameter tuning — GridSearchCV, RandomizedSearchCV, Bayesian Optimization (ALL models)
5. Stacking/Blending MUST (before and after tuning)
6. Accuracy, Precision, Recall, F1 for all models
7. Data Augmentation (SMOTE + SMOTETomek)
8. Normalized Confusion Matrices
9. Learning Curves
10. XAI — SHAP
11. Ablation Study

> Upload `HO_infxn_analysis.csv` to Colab, then Runtime → Run All

## Cell 1 — Install Libraries

In [ ]:
!pip install xgboost lightgbm imbalanced-learn shap scikit-optimize --quiet

## Cell 2 — Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time, re

# Text → features
from sklearn.feature_extraction.text import TfidfVectorizer

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, learning_curve
)
from sklearn.feature_selection import SelectKBest, chi2

# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

# Augmentation
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

# Bayesian optimization
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

# XAI
import shap

np.random.seed(42)
print("✅ All libraries loaded")

## Cell 3 — Load Dataset

In [ ]:
# ── CHANGE PATH IF NEEDED ─────────────────────────────
DATA_PATH = "HO_infxn_analysis.csv"
# Google Drive: DATA_PATH = "/content/drive/MyDrive/HO_infxn_analysis.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape  : {df.shape}")
print(f"Target distribution:")
print(df['group_binary'].value_counts())
print(f"\nClass balance  : {df['group_binary'].value_counts(normalize=True).round(3).to_dict()}")
print(f"\nSample columns : {df.columns.tolist()[:10]}")

## Cell 4 — Convert Tabular Data → Clinical Text Narratives
**This is the core step required by the professor.**
Every patient record (row) is converted into a natural language clinical text sentence.
This allows us to apply NLP-style preprocessing (TF-IDF) and ML models on text features.

In [ ]:
# =============================================================
# STEP 1: EXPLICIT DATA CLEANING (before text conversion)
# =============================================================
print("="*65)
print("STEP 1: DATA CLEANING")
print("="*65)

# 1a. Drop identifier / leakage columns
drop_cols = ["group", "group_index", "deidentified_patient_id",
             "deidentified_month", "deidentified_year"]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(f"1a. Dropped {len(drop_cols)} leakage/identifier columns")
print(f"    Remaining columns: {df.shape[1]}")

# 1b. Fix age: '>90' string → numeric 92
df["age"] = df["age"].replace(">90", "92").astype(float)
print(f"\n1b. age '>90' replaced with 92 → cast to float")
print(f"    Age range: {df['age'].min():.1f} – {df['age'].max():.1f}")

# 1c. Handle time_to_infxn (NaN for all controls — not infected)
df["time_to_infxn"] = pd.to_numeric(df["time_to_infxn"], errors="coerce")
median_tti = df["time_to_infxn"].median()
df["time_to_infxn"].fillna(median_tti, inplace=True)
print(f"\n1c. time_to_infxn NaN imputed with median = {median_tti:.2f} days")
print(f"    NaN remaining: {df['time_to_infxn'].isnull().sum()}")

# 1d. Check and report all missing values
missing = df.isnull().sum()
print(f"\n1d. Missing values after cleaning:")
print(f"    {missing[missing > 0].to_dict() if missing.sum() > 0 else 'None — clean dataset ✅'}")

# 1e. Separate target variable
y_all = df["group_binary"].values
X_df  = df.drop(columns=["group_binary"])
print(f"\n1e. Target separated: class 0={sum(y_all==0)}, class 1={sum(y_all==1)}")
print(f"    Features remaining: {X_df.shape[1]}")

# =============================================================
# STEP 2: CONVERT TABULAR DATA → CLINICAL TEXT (professor requirement)
# =============================================================
print("\n" + "="*65)
print("STEP 2: CONVERT TABULAR DATA → CLINICAL TEXT NARRATIVES")
print("="*65)
print("(Required: convert tabular data to text before all processing)")

def row_to_clinical_text(row, all_cols):
    """
    Convert one tabular patient record into a clinical text narrative.
    Every feature is embedded as a meaningful word/phrase in natural language.
    """
    age      = str(row.get("age", "0")).replace(">90","92")
    sex      = str(row.get("sex", "unknown"))
    duration = str(round(float(row.get("duration", 0)), 1))
    analysis = str(row.get("analysis", "unknown"))
    run_type = str(row.get("run", "unknown")).replace("_", " ")
    surgery  = "underwent surgery" if row.get("any_surgery", 0)==1 else "no surgery"
    try:
        elix = str(int(float(str(row.get("elix_index_mortality", 0)))))
    except:
        elix = "0"
    try:
        tti_val = row.get("time_to_infxn", 0)
        tti = str(round(float(tti_val), 1)) if not pd.isna(tti_val) else "0.0"
    except:
        tti = "0.0"

    abx_cols = [c for c in all_cols if "_0_60" in c]
    abx_list = [c.replace("_0_60","").replace("_"," ")
                for c in abx_cols if row.get(c, 0)==1]
    cp_cols  = ["DS_Entero_cp","ESBL_cp","CDiff_cp","VSE_cp","VRE_cp",
                "MSSA_cp","MRSA_cp","DS_PsA_cp","DR_PsA_cp"]
    paths    = [c.replace("_cp","").replace("_"," ")
                for c in cp_cols if row.get(c, 0)>0]
    elix_cols = [c for c in all_cols
                 if c.startswith("elix_") and c!="elix_index_mortality"]
    comorbids = [c.replace("elix_","").replace("_"," ")
                 for c in elix_cols if row.get(c, 0)==1]

    return (
        f"patient age {age} sex {sex} "
        f"admitted {duration} days "
        f"analysis {analysis} run {run_type} "
        f"time to infection {tti} days "
        f"{surgery} "
        f"elixhauser score {elix} "
        f"antibiotics {' '.join(abx_list) if abx_list else 'none'} "
        f"colonization {' '.join(paths) if paths else 'none'} "
        f"comorbidities {' '.join(comorbids) if comorbids else 'none'}"
    )

all_cols = X_df.columns.tolist()
texts    = [row_to_clinical_text(row, all_cols) for _, row in X_df.iterrows()]
labels   = y_all.tolist()

print(f"\nConverted {len(texts)} rows to clinical text narratives")
print(f"Label distribution: {pd.Series(labels).value_counts().to_dict()}")
print(f"\nExample text (patient 0):")
print(texts[0])
print(f"\nExample text (patient 1):")
print(texts[1])

## Cell 5 — ALL Preprocessing Techniques
Applied to the **text data** (not tabular features):
1. **Train/Test Split** — stratified 80/20
2. **TF-IDF Vectorization** — converts text to numerical matrix (max 5000 features)
3. **SelectKBest (chi²)** — feature selection: keep top 2000 most informative TF-IDF features
4. **StandardScaler** — normalize TF-IDF weights
5. **SMOTE subset for SVM** — 15k stratified subset (SVM is O(n²))
6. **20k tuning subset** — for hyperparameter search (standard practice for 96k rows)

## EDA - Exploratory Data Analysis on Text Data

In [ ]:
print("="*65)
print("EDA - EXPLORATORY DATA ANALYSIS")
print("="*65)

class_counts = pd.Series(labels).value_counts()
print(f"\nClass distribution: {class_counts.to_dict()}")
print(f"Imbalance ratio: {class_counts[0]/class_counts[1]:.2f}:1 (controls vs cases)")

text_lengths = [len(t.split()) for t in texts]
print(f"Text length - Mean={np.mean(text_lengths):.1f}  Std={np.std(text_lengths):.1f}  Min={min(text_lengths)}  Max={max(text_lengths)}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("EDA - Hospital-Onset Infection Dataset", fontsize=13, fontweight="bold")

# Class distribution
axes[0].bar(["Control (0)", "Case (1)"], [class_counts[0], class_counts[1]],
            color=["#2196F3","#F44336"], edgecolor="white")
axes[0].set_title("Class Distribution", fontweight="bold")
for i, v in enumerate([class_counts[0], class_counts[1]]):
    axes[0].text(i, v+500, f"{v:,}\n({v/len(labels)*100:.1f}%)", ha="center", fontsize=9)

# Text length histogram
len0 = [len(texts[i].split()) for i in range(len(texts)) if labels[i]==0]
len1 = [len(texts[i].split()) for i in range(len(texts)) if labels[i]==1]
axes[1].hist(len0, bins=30, alpha=0.6, color="#2196F3", label="Control")
axes[1].hist(len1, bins=30, alpha=0.6, color="#F44336", label="Case")
axes[1].set_title("Text Length Distribution", fontweight="bold")
axes[1].set_xlabel("Word count per text"); axes[1].legend()

# Top words in cases
from sklearn.feature_extraction.text import CountVectorizer
cv_eda = CountVectorizer(max_features=500, stop_words="english")
X_eda  = cv_eda.fit_transform(texts)
vocab  = np.array(cv_eda.get_feature_names_out())
X_c1   = X_eda[np.array(labels)==1].toarray().mean(0)
top10  = vocab[np.argsort(X_c1)[-10:][::-1]]
axes[2].barh(range(10), np.sort(X_c1)[-10:][::-1], color="#F44336")
axes[2].set_yticks(range(10)); axes[2].set_yticklabels(top10)
axes[2].set_title("Top Words in Cases", fontweight="bold"); axes[2].set_xlabel("Mean Term Freq")
plt.tight_layout(); plt.savefig("eda_overview.png", dpi=150); plt.show()

# Surgery rate by class
surg_rate_0 = sum(1 for t in [texts[i] for i in range(len(texts)) if labels[i]==0] if "underwent surgery" in t) / class_counts[0]
surg_rate_1 = sum(1 for t in [texts[i] for i in range(len(texts)) if labels[i]==1] if "underwent surgery" in t) / class_counts[1]
print(f"Surgery rate - Control: {surg_rate_0:.3f}  Case: {surg_rate_1:.3f}")
print("\nSaved: eda_overview.png")

In [ ]:
# =============================================================
# STEP 3: ALL PREPROCESSING TECHNIQUES ON TEXT DATA
# =============================================================
print("="*65)
print("STEP 3: PREPROCESSING PIPELINE")
print("="*65)
print("""
Preprocessing applied (ALL techniques):
  P1. Stratified Train/Test Split (80/20)
  P2. Text Normalization (lowercase, strip accents, tokenization)
  P3. TF-IDF Vectorization (unigrams + bigrams, log-TF, top 5000 terms)
  P4. Feature Selection — SelectKBest chi² (top 2000 features)
  P5. Feature Scaling — MaxAbsScaler (sparse-safe normalization)
  P6. Dense conversion for tree-based models
  P7. Tuning subset (20k stratified) & SVM subset (8k)
""")

import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split as tts

# P1. Stratified train/test split (80/20) ─────────────────────────────────
X_train_txt, X_test_txt, y_train, y_test = tts(
    texts, labels, test_size=0.20, stratify=labels, random_state=42
)
y_train_arr = np.array(y_train)
y_test_arr  = np.array(y_test)
print(f"P1. Stratified 80/20 split:")
print(f"    Train={len(X_train_txt)}  Test={len(X_test_txt)}")
print(f"    Train class dist: {np.bincount(y_train_arr)}")

# P2+P3. TF-IDF Vectorization (includes tokenization + normalization) ───────
print(f"\nP2+P3. TF-IDF Vectorization (fit on train only):")
tfidf = TfidfVectorizer(
    max_features=5000,        # top 5000 terms
    ngram_range=(1, 2),       # unigrams + bigrams
    sublinear_tf=True,        # log(1+tf) normalization
    min_df=5,                 # ignore very rare terms (< 5 docs)
    strip_accents="unicode",  # text normalization
    analyzer="word",          # word-level tokenization
    lowercase=True,           # convert to lowercase
)
X_train_tfidf = tfidf.fit_transform(X_train_txt)
X_test_tfidf  = tfidf.transform(X_test_txt)
print(f"    Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"    Train TF-IDF: {X_train_tfidf.shape}")
print(f"    Test  TF-IDF: {X_test_tfidf.shape}")

# P4. Feature Selection — SelectKBest chi² ─────────────────────────────────
print(f"\nP4. SelectKBest chi² (k=2000):")
selector = SelectKBest(chi2, k=2000)
X_train_sel = selector.fit_transform(X_train_tfidf, y_train_arr)
X_test_sel  = selector.transform(X_test_tfidf)
print(f"    Selected features: {X_train_sel.shape[1]} / {X_train_tfidf.shape[1]}")

# P5. MaxAbsScaler (sparse-safe normalization) ─────────────────────────────
print(f"\nP5. MaxAbsScaler (fit on train only):")
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train_sel)
X_test_scaled  = scaler.transform(X_test_sel)
print(f"    Scaled to [-1, 1] range (preserves sparsity)")

# P6. Convert to dense for tree-based models ───────────────────────────────
X_train_dense = X_train_scaled.toarray() if sp.issparse(X_train_scaled) else X_train_scaled
X_test_dense  = X_test_scaled.toarray()  if sp.issparse(X_test_scaled)  else X_test_scaled
print(f"\nP6. Dense conversion:")
print(f"    Train matrix: {X_train_dense.shape}  ({X_train_dense.nbytes / 1e6:.1f} MB)")
print(f"    Test  matrix: {X_test_dense.shape}")

# P7. Tuning subset and SVM subset ─────────────────────────────────────────
X_tune, _, y_tune, _ = tts(X_train_dense, y_train_arr,
                             train_size=20000, stratify=y_train_arr, random_state=42)
X_svm,  _, y_svm,  _ = tts(X_train_dense, y_train_arr,
                             train_size=8000,  stratify=y_train_arr, random_state=42)
print(f"\nP7. Subsets for tuning:")
print(f"    Tuning subset: {X_tune.shape}  (used for GridSearch/RandomSearch/Bayes)")
print(f"    SVM subset:    {X_svm.shape}   (SVM is O(n²) — subset for speed)")
print(f"\n✅ All preprocessing complete")

# ── OUTLIER DETECTION on TF-IDF document norms (IQR) ────────────────────────
print(f"\nOUTLIER DETECTION - IQR on TF-IDF document norms:")
doc_norms = np.array(X_train_tfidf.power(2).sum(axis=1)).flatten()
q1_n = np.percentile(doc_norms, 25)
q3_n = np.percentile(doc_norms, 75)
iqr_n = q3_n - q1_n
lower_bound = q1_n - 1.5 * iqr_n
upper_bound = q3_n + 1.5 * iqr_n
n_outliers = ((doc_norms < lower_bound) | (doc_norms > upper_bound)).sum()
print(f"    Q1={q1_n:.4f}  Q3={q3_n:.4f}  IQR={iqr_n:.4f}")
print(f"    Bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"    Outlier documents: {n_outliers} ({n_outliers/len(doc_norms)*100:.2f}%)")
print(f"    Action: Retained (removing clinical text docs loses patient data)")

## Cell 6 — Helper Functions

In [ ]:
results_before = []
results_after  = []
results_grid   = []
results_rand   = []
results_bayes  = []
results_aug    = []

cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def evaluate(name, model, Xtr, ytr, store_list):
    """Train model, evaluate on full test set, store all metrics."""
    t0 = time.time()
    model.fit(Xtr, ytr)
    elapsed = round(time.time()-t0, 2)
    pred  = model.predict(X_test_dense)
    proba = (model.predict_proba(X_test_dense)[:,1]
             if hasattr(model,"predict_proba") else np.zeros(len(y_test_arr)))
    row = {
        "Model"    : name,
        "Accuracy" : round(accuracy_score(y_test_arr, pred), 4),
        "Precision": round(precision_score(y_test_arr, pred, average="weighted", zero_division=0), 4),
        "Recall"   : round(recall_score(y_test_arr, pred,    average="weighted", zero_division=0), 4),
        "F1"       : round(f1_score(y_test_arr, pred,        average="weighted", zero_division=0), 4),
        "ROC_AUC"  : round(roc_auc_score(y_test_arr, proba) if proba.sum()>0 else 0.0, 4),
        "Train_s"  : elapsed,
    }
    store_list.append(row)
    print(f"  {name:<45}  Acc={row['Accuracy']:.4f}  F1={row['F1']:.4f}  AUC={row['ROC_AUC']:.4f}  ({elapsed}s)")
    return model, pred

def run_search(label, search_obj, Xtr_, ytr_, store_list, tag=""):
    """Run a search object, retrain best on full X_train, evaluate on test."""
    t0 = time.time()
    search_obj.fit(Xtr_, ytr_)
    elapsed = round(time.time()-t0, 1)
    # Retrain best estimator on full training set
    best = search_obj.best_estimator_
    best.fit(X_train_dense, y_train_arr)
    pred  = best.predict(X_test_dense)
    proba = (best.predict_proba(X_test_dense)[:,1]
             if hasattr(best,"predict_proba") else np.zeros(len(y_test_arr)))
    row = {
        "Model"      : label,
        "Method"     : tag,
        "Best_Params": str(search_obj.best_params_),
        "CV_F1"      : round(search_obj.best_score_, 4),
        "Accuracy"   : round(accuracy_score(y_test_arr, pred), 4),
        "Precision"  : round(precision_score(y_test_arr, pred, average="weighted", zero_division=0), 4),
        "Recall"     : round(recall_score(y_test_arr, pred,    average="weighted", zero_division=0), 4),
        "F1"         : round(f1_score(y_test_arr, pred,        average="weighted", zero_division=0), 4),
        "ROC_AUC"    : round(roc_auc_score(y_test_arr, proba) if proba.sum()>0 else 0.0, 4),
        "Time_s"     : elapsed,
    }
    store_list.append(row)
    print(f"    {tag:<22}  CV_F1={search_obj.best_score_:.4f}  Test_F1={row['F1']:.4f}  ({elapsed}s)")
    print(f"    Best params: {search_obj.best_params_}")
    return search_obj, best

def run_bayes(label, estimator, search_spaces, Xtr_, ytr_, store_list, n_iter=20):
    """Run BayesSearchCV, retrain best on full X_train, evaluate on test."""
    t0 = time.time()
    bayes = BayesSearchCV(estimator, search_spaces=search_spaces,
                          n_iter=n_iter, cv=cv3, scoring="f1_weighted",
                          n_jobs=-1, random_state=42, verbose=0)
    bayes.fit(Xtr_, ytr_)
    elapsed = round(time.time()-t0, 1)
    best = bayes.best_estimator_
    best.fit(X_train_dense, y_train_arr)
    pred  = best.predict(X_test_dense)
    proba = (best.predict_proba(X_test_dense)[:,1]
             if hasattr(best,"predict_proba") else np.zeros(len(y_test_arr)))
    row = {
        "Model"      : label,
        "Method"     : "BayesSearchCV",
        "Best_Params": str(dict(bayes.best_params_)),
        "CV_F1"      : round(bayes.best_score_, 4),
        "Accuracy"   : round(accuracy_score(y_test_arr, pred), 4),
        "Precision"  : round(precision_score(y_test_arr, pred, average="weighted", zero_division=0), 4),
        "Recall"     : round(recall_score(y_test_arr, pred,    average="weighted", zero_division=0), 4),
        "F1"         : round(f1_score(y_test_arr, pred,        average="weighted", zero_division=0), 4),
        "ROC_AUC"    : round(roc_auc_score(y_test_arr, proba) if proba.sum()>0 else 0.0, 4),
        "Time_s"     : elapsed,
    }
    store_list.append(row)
    print(f"    BayesSearchCV      CV_F1={bayes.best_score_:.4f}  Test_F1={row['F1']:.4f}  ({elapsed}s)")
    print(f"    Best params: {dict(bayes.best_params_)}")
    return bayes, best

print("✅ Helper functions ready")

## Cell 7 — ML Models WITHOUT Hyperparameter Tuning (9 Models + Stacking MUST)

In [ ]:
print("="*70)
print("ML MODELS — BEFORE HYPERPARAMETER TUNING")
print("All trained on TF-IDF text features from clinical narratives")
print("="*70)

# 1. Decision Tree
evaluate("Decision Tree", DecisionTreeClassifier(random_state=42),
         X_train_dense, y_train_arr, results_before)

# 2. Logistic Regression
evaluate("Logistic Regression", LogisticRegression(max_iter=500, random_state=42),
         X_train_dense, y_train_arr, results_before)

# 3. SVM (LinearSVC — fast on TF-IDF, wrapped for predict_proba)
svm_base = CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2000))
evaluate("SVM (LinearSVC)", svm_base, X_train_dense, y_train_arr, results_before)

# 4. Random Forest
evaluate("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
         X_train_dense, y_train_arr, results_before)

# 5. XGBoost
evaluate("XGBoost", XGBClassifier(n_estimators=100, eval_metric="logloss",
                                   random_state=42, verbosity=0, n_jobs=-1),
         X_train_dense, y_train_arr, results_before)

# 6. LightGBM
evaluate("LightGBM", LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1, n_jobs=-1),
         X_train_dense, y_train_arr, results_before)

# 7. AdaBoost
evaluate("AdaBoost", AdaBoostClassifier(n_estimators=50, random_state=42),
         X_train_dense, y_train_arr, results_before)

# 8. Voting Ensemble (RF + XGB + LGB)
vote_base = VotingClassifier(
    estimators=[
        ("rf",  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        ("xgb", XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1)),
        ("lgb", LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1, n_jobs=-1)),
    ], voting="soft", n_jobs=-1)
evaluate("Voting Ensemble (RF+XGB+LGB)", vote_base, X_train_dense, y_train_arr, results_before)

# 9. STACKING / BLENDING — MUST (before tuning)
stack_base = StackingClassifier(
    estimators=[
        ("dt",  DecisionTreeClassifier(max_depth=15, random_state=42)),
        ("lr",  LogisticRegression(max_iter=300, random_state=42)),
        ("rf",  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
        ("xgb", XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1)),
        ("lgb", LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1, n_jobs=-1)),
    ],
    final_estimator=LogisticRegression(max_iter=500, random_state=42),
    cv=3, n_jobs=-1
)
evaluate("Stacking (CV-based) - MUST", stack_base, X_train_dense, y_train_arr, results_before)

# Print results table
print("\n" + "="*70)
print("TABLE: ML MODELS BEFORE TUNING (on TF-IDF text features)")
print("="*70)
df_before = pd.DataFrame(results_before)
print(df_before.to_string(index=False))
df_before.to_csv("results_before_tuning.csv", index=False)
print("\nSaved: results_before_tuning.csv")

# ── BLENDING (holdout-based - DIFFERENT from Stacking) ───────────────────────
print("\n--- BLENDING (fixed holdout generates meta-features) - MUST ---")
print("    Key difference from Stacking: uses a held-out validation set, NOT cross-validation")
X_bl_tr, X_holdout, y_bl_tr, y_holdout = train_test_split(
    X_train_dense, y_train_arr, test_size=0.20, stratify=y_train_arr, random_state=42)
blend_bases = {
    "dt" : DecisionTreeClassifier(max_depth=15, random_state=42),
    "lr" : LogisticRegression(max_iter=300, random_state=42),
    "rf" : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "xgb": XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1),
    "lgb": LGBMClassifier(n_estimators=100, random_state=42, verbosity=-1, n_jobs=-1),
}
for nm, bm in blend_bases.items():
    bm.fit(X_bl_tr, y_bl_tr)
meta_holdout = np.column_stack([m.predict_proba(X_holdout)[:,1] for m in blend_bases.values()])
meta_test    = np.column_stack([m.predict_proba(X_test_dense)[:,1] for m in blend_bases.values()])
blend_meta   = LogisticRegression(C=1.0, max_iter=500, random_state=42)
blend_meta.fit(meta_holdout, y_holdout)
bp = blend_meta.predict(meta_test)
bproba = blend_meta.predict_proba(meta_test)[:,1]
blend_row = {
    "Model"    : "Blending (holdout-based) - MUST",
    "Accuracy" : round(accuracy_score(y_test_arr, bp), 4),
    "Precision": round(precision_score(y_test_arr, bp, average="weighted", zero_division=0), 4),
    "Recall"   : round(recall_score(y_test_arr, bp,    average="weighted", zero_division=0), 4),
    "F1"       : round(f1_score(y_test_arr, bp,        average="weighted", zero_division=0), 4),
    "ROC_AUC"  : round(roc_auc_score(y_test_arr, bproba), 4),
    "Train_s"  : 0.0
}
results_before.append(blend_row)
print(f"  {'Blending (holdout-based) - MUST':<48} Acc={blend_row['Accuracy']:.4f} F1={blend_row['F1']:.4f} AUC={blend_row['ROC_AUC']:.4f}")

## Cell 8 — Hyperparameter Tuning: GridSearchCV + RandomizedSearchCV + BayesSearchCV
All 3 methods applied to ALL 6 models on 20k tuning subset.
Final best models retrained on full 96k training set.

In [ ]:
print("="*70)
print("HYPERPARAMETER TUNING — 3 METHODS ON ALL 6 MODELS")
print("GridSearchCV  |  RandomizedSearchCV  |  BayesSearchCV")
print("="*70)

# =============================================================
# 1. DECISION TREE
# =============================================================
print("\n[1/6] DECISION TREE")
dt_grid, dt_grid_best = run_search("Decision Tree",
    GridSearchCV(DecisionTreeClassifier(random_state=42),
        {"max_depth":[5,10,20,None],"min_samples_split":[2,5,10],"criterion":["gini","entropy"]},
        cv=cv3, scoring="f1_weighted", n_jobs=-1),
    X_tune, y_tune, results_grid, "GridSearchCV")

dt_rand, dt_rand_best = run_search("Decision Tree",
    RandomizedSearchCV(DecisionTreeClassifier(random_state=42),
        {"max_depth":[3,5,8,10,15,20,None],"min_samples_split":[2,5,10,20],
         "min_samples_leaf":[1,2,5],"criterion":["gini","entropy"],"splitter":["best","random"]},
        n_iter=20, cv=cv3, scoring="f1_weighted", n_jobs=-1, random_state=42),
    X_tune, y_tune, results_rand, "RandomizedSearchCV")

dt_bayes, dt_bayes_best = run_bayes("Decision Tree",
    DecisionTreeClassifier(random_state=42),
    {"max_depth":Integer(2,30),"min_samples_split":Integer(2,20),
     "min_samples_leaf":Integer(1,10),"criterion":Categorical(["gini","entropy"])},
    X_tune, y_tune, results_bayes, n_iter=20)

# Pick best DT across all 3 methods
dt_scores = [dt_grid.best_score_, dt_rand.best_score_, dt_bayes.best_score_]
dt_bests  = [dt_grid_best, dt_rand_best, dt_bayes_best]
dt_final  = dt_bests[np.argmax(dt_scores)]
print(f"  ★ Best DT: {'GridSearch' if np.argmax(dt_scores)==0 else 'RandomSearch' if np.argmax(dt_scores)==1 else 'Bayesian'}")

# =============================================================
# 2. LOGISTIC REGRESSION
# =============================================================
print("\n[2/6] LOGISTIC REGRESSION")
lr_grid, lr_grid_best = run_search("Logistic Regression",
    GridSearchCV(LogisticRegression(max_iter=500, random_state=42),
        {"C":[0.01,0.1,1,10],"solver":["liblinear","lbfgs"]},
        cv=cv3, scoring="f1_weighted", n_jobs=-1),
    X_tune, y_tune, results_grid, "GridSearchCV")

lr_rand, lr_rand_best = run_search("Logistic Regression",
    RandomizedSearchCV(LogisticRegression(max_iter=1000, random_state=42),
        {"C":[0.001,0.01,0.1,0.5,1,5,10,50,100],"solver":["liblinear","lbfgs","saga"]},
        n_iter=20, cv=cv3, scoring="f1_weighted", n_jobs=-1, random_state=42),
    X_tune, y_tune, results_rand, "RandomizedSearchCV")

lr_bayes, lr_bayes_best = run_bayes("Logistic Regression",
    LogisticRegression(max_iter=1000, random_state=42),
    {"C":Real(0.001,100,prior="log-uniform"),"solver":Categorical(["liblinear","lbfgs","saga"])},
    X_tune, y_tune, results_bayes, n_iter=20)

lr_scores = [lr_grid.best_score_, lr_rand.best_score_, lr_bayes.best_score_]
lr_bests  = [lr_grid_best, lr_rand_best, lr_bayes_best]
lr_final  = lr_bests[np.argmax(lr_scores)]
print(f"  ★ Best LR: {'GridSearch' if np.argmax(lr_scores)==0 else 'RandomSearch' if np.argmax(lr_scores)==1 else 'Bayesian'}")

# =============================================================
# 3. SVM  (8k subset)
# =============================================================
print("\n[3/6] SVM (LinearSVC — 8k subset for speed)")
svm_grid, svm_grid_best = run_search("SVM",
    GridSearchCV(CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42)),
        {"base_estimator__C":[0.01,0.1,1,10]},
        cv=cv3, scoring="f1_weighted", n_jobs=-1),
    X_svm, y_svm, results_grid, "GridSearchCV")

svm_rand, svm_rand_best = run_search("SVM",
    RandomizedSearchCV(CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42)),
        {"base_estimator__C":[0.001,0.01,0.1,1,10,100]},
        n_iter=10, cv=cv3, scoring="f1_weighted", n_jobs=-1, random_state=42),
    X_svm, y_svm, results_rand, "RandomizedSearchCV")

svm_bayes, svm_bayes_best = run_bayes("SVM",
    CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42)),
    {"base_estimator__C":Real(0.001,100,prior="log-uniform")},
    X_svm, y_svm, results_bayes, n_iter=15)

svm_scores = [svm_grid.best_score_, svm_rand.best_score_, svm_bayes.best_score_]
svm_bests  = [svm_grid_best, svm_rand_best, svm_bayes_best]
svm_final  = svm_bests[np.argmax(svm_scores)]
print(f"  ★ Best SVM: {'GridSearch' if np.argmax(svm_scores)==0 else 'RandomSearch' if np.argmax(svm_scores)==1 else 'Bayesian'}")

# =============================================================
# 4. RANDOM FOREST
# =============================================================
print("\n[4/6] RANDOM FOREST")
rf_grid, rf_grid_best = run_search("Random Forest",
    GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200],"max_depth":[10,20,None],"max_features":["sqrt","log2"]},
        cv=cv3, scoring="f1_weighted", n_jobs=-1),
    X_tune, y_tune, results_grid, "GridSearchCV")

rf_rand, rf_rand_best = run_search("Random Forest",
    RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,150,200,250],"max_depth":[5,10,15,20,None],
         "max_features":["sqrt","log2"],"min_samples_leaf":[1,2,4],"bootstrap":[True,False]},
        n_iter=20, cv=cv3, scoring="f1_weighted", n_jobs=-1, random_state=42),
    X_tune, y_tune, results_rand, "RandomizedSearchCV")

rf_bayes, rf_bayes_best = run_bayes("Random Forest",
    RandomForestClassifier(random_state=42, n_jobs=-1),
    {"n_estimators":Integer(50,300),"max_depth":Integer(5,30),
     "max_features":Categorical(["sqrt","log2"]),"min_samples_leaf":Integer(1,8)},
    X_tune, y_tune, results_bayes, n_iter=20)

rf_scores = [rf_grid.best_score_, rf_rand.best_score_, rf_bayes.best_score_]
rf_bests  = [rf_grid_best, rf_rand_best, rf_bayes_best]
rf_final  = rf_bests[np.argmax(rf_scores)]
print(f"  ★ Best RF: {'GridSearch' if np.argmax(rf_scores)==0 else 'RandomSearch' if np.argmax(rf_scores)==1 else 'Bayesian'}")

# =============================================================
# 5. XGBOOST
# =============================================================
print("\n[5/6] XGBOOST")
xgb_grid, xgb_grid_best = run_search("XGBoost",
    GridSearchCV(XGBClassifier(eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1),
        {"n_estimators":[100,200,300],"max_depth":[3,5,7],"learning_rate":[0.05,0.1,0.2]},
        cv=cv3, scoring="f1_weighted", n_jobs=-1),
    X_tune, y_tune, results_grid, "GridSearchCV")

xgb_rand, xgb_rand_best = run_search("XGBoost",
    RandomizedSearchCV(XGBClassifier(eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1),
        {"n_estimators":[100,200,300,400],"max_depth":[3,4,5,6,7],
         "learning_rate":[0.01,0.05,0.1,0.15,0.2],"subsample":[0.7,0.8,1.0],
         "colsample_bytree":[0.7,0.8,1.0],"min_child_weight":[1,3,5]},
        n_iter=20, cv=cv3, scoring="f1_weighted", n_jobs=-1, random_state=42),
    X_tune, y_tune, results_rand, "RandomizedSearchCV")

xgb_bayes, xgb_bayes_best = run_bayes("XGBoost",
    XGBClassifier(eval_metric="logloss", random_state=42, verbosity=0, n_jobs=-1),
    {"n_estimators":Integer(100,400),"max_depth":Integer(3,10),
     "learning_rate":Real(0.01,0.3,prior="log-uniform"),
     "subsample":Real(0.6,1.0),"colsample_bytree":Real(0.6,1.0)},
    X_tune, y_tune, results_bayes, n_iter=20)

xgb_scores = [xgb_grid.best_score_, xgb_rand.best_score_, xgb_bayes.best_score_]
xgb_bests  = [xgb_grid_best, xgb_rand_best, xgb_bayes_best]
xgb_final  = xgb_bests[np.argmax(xgb_scores)]
print(f"  ★ Best XGB: {'GridSearch' if np.argmax(xgb_scores)==0 else 'RandomSearch' if np.argmax(xgb_scores)==1 else 'Bayesian'}")

# =============================================================
# 6. LIGHTGBM
# =============================================================
print("\n[6/6] LIGHTGBM")
lgb_grid, lgb_grid_best = run_search("LightGBM",
    GridSearchCV(LGBMClassifier(random_state=42, verbosity=-1, n_jobs=-1),
        {"n_estimators":[100,200,300],"num_leaves":[31,63,127],"learning_rate":[0.05,0.1,0.2]},
        cv=cv3, scoring="f1_weighted", n_jobs=-1),
    X_tune, y_tune, results_grid, "GridSearchCV")

lgb_rand, lgb_rand_best = run_search("LightGBM",
    RandomizedSearchCV(LGBMClassifier(random_state=42, verbosity=-1, n_jobs=-1),
        {"n_estimators":[100,200,300,400],"num_leaves":[31,63,95,127],
         "learning_rate":[0.01,0.05,0.1,0.2],"min_child_samples":[10,20,50,100],
         "subsample":[0.7,0.8,1.0],"reg_alpha":[0,0.1,0.5],"reg_lambda":[0,0.1,1.0]},
        n_iter=20, cv=cv3, scoring="f1_weighted", n_jobs=-1, random_state=42),
    X_tune, y_tune, results_rand, "RandomizedSearchCV")

lgb_bayes, lgb_bayes_best = run_bayes("LightGBM",
    LGBMClassifier(random_state=42, verbosity=-1, n_jobs=-1),
    {"n_estimators":Integer(100,400),"num_leaves":Integer(20,150),
     "learning_rate":Real(0.01,0.3,prior="log-uniform"),
     "min_child_samples":Integer(10,100),"subsample":Real(0.6,1.0),
     "reg_alpha":Real(0.0,1.0),"reg_lambda":Real(0.0,1.0)},
    X_tune, y_tune, results_bayes, n_iter=20)

lgb_scores = [lgb_grid.best_score_, lgb_rand.best_score_, lgb_bayes.best_score_]
lgb_bests  = [lgb_grid_best, lgb_rand_best, lgb_bayes_best]
lgb_final  = lgb_bests[np.argmax(lgb_scores)]
print(f"  ★ Best LGB: {'GridSearch' if np.argmax(lgb_scores)==0 else 'RandomSearch' if np.argmax(lgb_scores)==1 else 'Bayesian'}")

# =============================================================
# STACKING TUNED — MUST (after tuning)
# =============================================================
print("\nSTACKING / BLENDING TUNED — MUST (all best estimators)")
stack_tuned = StackingClassifier(
    estimators=[("dt", dt_final), ("lr", lr_final), ("svm", svm_final),
                ("rf", rf_final), ("xgb", xgb_final), ("lgb", lgb_final)],
    final_estimator=LogisticRegression(C=10, max_iter=500, random_state=42),
    cv=3, n_jobs=-1
)
stack_tuned.fit(X_train_dense, y_train_arr)
stk_pred  = stack_tuned.predict(X_test_dense)
stk_proba = stack_tuned.predict_proba(X_test_dense)[:,1]
stk_row = {"Model":"★ Stacking Tuned [MUST]",
    "Accuracy":round(accuracy_score(y_test_arr,stk_pred),4),
    "Precision":round(precision_score(y_test_arr,stk_pred,average="weighted",zero_division=0),4),
    "Recall":round(recall_score(y_test_arr,stk_pred,average="weighted",zero_division=0),4),
    "F1":round(f1_score(y_test_arr,stk_pred,average="weighted",zero_division=0),4),
    "ROC_AUC":round(roc_auc_score(y_test_arr,stk_proba),4)}
results_after.append(stk_row)
print(f"  Stacking Tuned:  Acc={stk_row['Accuracy']}  F1={stk_row['F1']}  AUC={stk_row['ROC_AUC']}")

# =============================================================
# PRINT ALL TABLES
# =============================================================
print("\n" + "="*70)
print("TABLE A — GridSearchCV Results (ALL 6 Models)")
print("="*70)
df_grid = pd.DataFrame(results_grid)
print(df_grid[["Model","Best_Params","CV_F1","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))

print("\n" + "="*70)
print("TABLE B — RandomizedSearchCV Results (ALL 6 Models)")
print("="*70)
df_rand = pd.DataFrame(results_rand)
print(df_rand[["Model","Best_Params","CV_F1","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))

print("\n" + "="*70)
print("TABLE C — BayesSearchCV Results (ALL 6 Models)")
print("="*70)
df_bayes = pd.DataFrame(results_bayes)
print(df_bayes[["Model","Best_Params","CV_F1","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))

print("\n" + "="*70)
print("TABLE D — Best Method Per Model + Stacking Tuned MUST")
print("="*70)
best_rows = []
pairs = [("Decision Tree",dt_grid,dt_rand,dt_bayes,dt_final),
         ("Logistic Regression",lr_grid,lr_rand,lr_bayes,lr_final),
         ("SVM",svm_grid,svm_rand,svm_bayes,svm_final),
         ("Random Forest",rf_grid,rf_rand,rf_bayes,rf_final),
         ("XGBoost",xgb_grid,xgb_rand,xgb_bayes,xgb_final),
         ("LightGBM",lgb_grid,lgb_rand,lgb_bayes,lgb_final)]
for mname,g,r,b,best_est in pairs:
    s = [g.best_score_,r.best_score_,b.best_score_]
    winner = ["GridSearchCV","RandomizedSearchCV","BayesSearchCV"][np.argmax(s)]
    pred = best_est.predict(X_test_dense)
    proba = best_est.predict_proba(X_test_dense)[:,1] if hasattr(best_est,"predict_proba") else np.zeros(len(y_test_arr))
    best_rows.append({"Model":mname,"Best_Method":winner,
        "Accuracy":round(accuracy_score(y_test_arr,pred),4),
        "Precision":round(precision_score(y_test_arr,pred,average="weighted",zero_division=0),4),
        "Recall":round(recall_score(y_test_arr,pred,average="weighted",zero_division=0),4),
        "F1":round(f1_score(y_test_arr,pred,average="weighted",zero_division=0),4),
        "ROC_AUC":round(roc_auc_score(y_test_arr,proba),4)})
best_rows.append({**stk_row,"Best_Method":"All best estimators combined"})
df_best = pd.DataFrame(best_rows)
print(df_best.to_string(index=False))

print("\n" + "="*70)
print("TABLE E — All 3 Tuning Methods Side-by-Side per Model")
print("="*70)
compare_rows = []
for mname,g,r,b,_ in pairs:
    g_row = next(x for x in results_grid  if x["Model"]==mname)
    r_row = next(x for x in results_rand  if x["Model"]==mname)
    b_row = next(x for x in results_bayes if x["Model"]==mname)
    compare_rows.append({"Model":mname,
        "GridSearch_F1":g_row["F1"],"RandomSearch_F1":r_row["F1"],"Bayesian_F1":b_row["F1"],
        "GridSearch_AUC":g_row["ROC_AUC"],"RandomSearch_AUC":r_row["ROC_AUC"],"Bayesian_AUC":b_row["ROC_AUC"]})
df_compare = pd.DataFrame(compare_rows)
print(df_compare.to_string(index=False))

# Save all
df_grid.to_csv("results_gridsearch.csv",    index=False)
df_rand.to_csv("results_randomsearch.csv",  index=False)
df_bayes.to_csv("results_bayesian.csv",     index=False)
df_best.to_csv("results_after_tuning.csv",  index=False)
df_compare.to_csv("results_tuning_comparison.csv", index=False)
print("\nSaved: all 5 tuning CSV files")

# ── BLENDING TUNED (holdout-based, using best tuned estimators) - MUST ──────
print("\n--- BLENDING TUNED (holdout-based, tuned estimators) - MUST ---")
X_blt, X_bhl, y_blt, y_bhl = train_test_split(
    X_train_dense, y_train_arr, test_size=0.20, stratify=y_train_arr, random_state=42)
tbl_b = {"dt":dt_final, "lr":lr_final, "rf":rf_final, "xgb":xgb_final, "lgb":lgb_final}
for nm, bm in tbl_b.items():
    bm.fit(X_blt, y_blt)
meta_bhl = np.column_stack([m.predict_proba(X_bhl)[:,1] for m in tbl_b.values()])
meta_bts = np.column_stack([m.predict_proba(X_test_dense)[:,1] for m in tbl_b.values()])
bmt = LogisticRegression(C=10, max_iter=500, random_state=42)
bmt.fit(meta_bhl, y_bhl)
bt_p = bmt.predict(meta_bts)
bt_pr = bmt.predict_proba(meta_bts)[:,1]
blt_row = {
    "Model"    : "Blending Tuned (holdout) - MUST",
    "Best_Method": "Tuned base estimators + LR meta-learner on holdout",
    "Accuracy" : round(accuracy_score(y_test_arr, bt_p), 4),
    "Precision": round(precision_score(y_test_arr, bt_p, average="weighted", zero_division=0), 4),
    "Recall"   : round(recall_score(y_test_arr, bt_p,    average="weighted", zero_division=0), 4),
    "F1"       : round(f1_score(y_test_arr, bt_p,        average="weighted", zero_division=0), 4),
    "ROC_AUC"  : round(roc_auc_score(y_test_arr, bt_pr), 4),
}
results_after.append(blt_row)
print(f"  Blending Tuned: Acc={blt_row['Accuracy']} F1={blt_row['F1']} AUC={blt_row['ROC_AUC']}")
best_rows.append(blt_row)
df_best = pd.DataFrame(best_rows)
print("\n--- UPDATED TABLE D (incl. Stacking Tuned + Blending Tuned) ---")
print(df_best.to_string(index=False))

## Cell 9 — Before vs After Tuning Comparison Plot

In [ ]:
# Before vs After Tuning — F1 comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Hospital-Onset Infection (TF-IDF Text Features) — Before vs After Tuning",
             fontsize=14, fontweight="bold")

df_before_sorted = df_before.sort_values("F1")
df_best_sorted   = df_best.sort_values("F1")

for ax, df_, title in [
    (axes[0], df_before_sorted, "BEFORE Tuning — 9 Models"),
    (axes[1], df_best_sorted,   "AFTER Tuning — Best per Model + Stacking MUST"),
]:
    clrs = plt.cm.RdYlGn(np.linspace(0.25, 0.9, len(df_)))
    bars = ax.barh(df_.Model, df_.F1, color=clrs, edgecolor="white", height=0.6)
    for bar, val in zip(bars, df_.F1):
        ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=9)
    ax.set_xlim(0.5, 1.0); ax.set_xlabel("Weighted F1 Score")
    ax.set_title(title, fontsize=12, fontweight="bold"); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("model_comparison_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

# All 3 tuning methods comparison plot (fixed loop)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("All 3 Tuning Methods — F1 & AUC per Model", fontsize=14, fontweight="bold")
x = np.arange(len(df_compare)); w = 0.25
colors_3  = ["#2196F3","#FF9800","#4CAF50"]
methods_3 = ["GridSearchCV","RandomizedSearchCV","BayesSearchCV"]
f1_cols  = ["GridSearch_F1",  "RandomSearch_F1",  "Bayesian_F1"]
auc_cols = ["GridSearch_AUC", "RandomSearch_AUC", "Bayesian_AUC"]

for ax, col_list, ylabel in [(axes[0], f1_cols, "Weighted F1"), (axes[1], auc_cols, "ROC-AUC")]:
    for j, (col, color, label) in enumerate(zip(col_list, colors_3, methods_3)):
        bars = ax.bar(x + j*w - w, df_compare[col], w,
                      label=label, color=color, edgecolor="white")
        for bar, val in zip(bars, df_compare[col]):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels(df_compare.Model, rotation=15, ha="right")
    ax.set_ylabel(ylabel); ax.legend(); ax.grid(axis="y", alpha=0.3)

axes[0].set_title("F1 Score Comparison", fontweight="bold")
axes[1].set_title("ROC-AUC Comparison",  fontweight="bold")
plt.tight_layout()
plt.savefig("tuning_methods_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: model_comparison_before_after.png, tuning_methods_comparison.png")

## Combined Results: ALL Models — Before & After Tuning
Clean summary of Accuracy, Precision, Recall, F1 for ALL models in both phases.

In [ ]:
# =============================================================
# COMBINED RESULTS TABLE: BEFORE + AFTER (all metrics clearly)
# =============================================================
print("="*70)
print("COMBINED RESULTS — BEFORE vs AFTER HYPERPARAMETER TUNING")
print("Accuracy | Precision | Recall | F1 | ROC-AUC — ALL MODELS")
print("="*70)

print("\n--- BEFORE TUNING (9 models, no hyperparameter optimization) ---")
pd.set_option("display.width", 130)
print(df_before[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))

print("\n--- AFTER TUNING (GridSearchCV + RandomizedSearchCV + BayesSearchCV) ---")
print("--- Best method selected per model + Stacking/Blending MUST ---")
print(df_best[["Model","Best_Method","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))

# Improvement summary
print("\n--- IMPROVEMENT SUMMARY (Before → After) ---")
before_models = ["Decision Tree","Logistic Regression","SVM","Random Forest","XGBoost","LightGBM"]
for mname in before_models:
    b_row = df_before.loc[df_before.Model==mname]
    a_row = df_best.loc[df_best.Model==mname]
    if not b_row.empty and not a_row.empty:
        delta_f1  = a_row.F1.values[0]  - b_row.F1.values[0]
        delta_acc = a_row.Accuracy.values[0] - b_row.Accuracy.values[0]
        print(f"  {mname:<22}  F1: {b_row.F1.values[0]:.4f} → {a_row.F1.values[0]:.4f}  "
              f"(Δ={delta_f1:+.4f})   Acc: Δ={delta_acc:+.4f}")

# Save combined
df_combined = pd.concat([
    df_before[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].assign(Phase="Before_Tuning"),
    df_best[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].assign(Phase="After_Tuning")
], ignore_index=True)
df_combined.to_csv("results_combined.csv", index=False)
print("\nSaved: results_combined.csv")

## Cell 10 — Data Augmentation (SMOTE + SMOTETomek)

In [ ]:
print("="*65)
print("DATA AUGMENTATION — SMOTE & SMOTETomek on TF-IDF text features")
print("="*65)
print("(Required: apply augmentation and find results similar to #3 baseline)")

# SMOTE
X_smote, y_smote = SMOTE(random_state=42).fit_resample(X_train_dense, y_train_arr)
print(f"SMOTE balanced   : {np.bincount(y_smote)}")

# SMOTETomek
X_st, y_st = SMOTETomek(random_state=42).fit_resample(X_train_dense, y_train_arr)
print(f"SMOTETomek       : {np.bincount(y_st)}")

aug_rows = []

# LGB + SMOTE
m1 = LGBMClassifier(**{kk:vv for kk,vv in lgb_bayes.best_params_.items()},
                    random_state=42, verbosity=-1, n_jobs=-1)
evaluate("LightGBM + SMOTE", m1, X_smote, y_smote, aug_rows)

# LGB + SMOTETomek
m2 = LGBMClassifier(**{kk:vv for kk,vv in lgb_bayes.best_params_.items()},
                    random_state=42, verbosity=-1, n_jobs=-1)
evaluate("LightGBM + SMOTETomek", m2, X_st, y_st, aug_rows)

# RF + SMOTE
m3 = RandomForestClassifier(**{kk:vv for kk,vv in rf_bayes.best_params_.items()},
                             random_state=42, n_jobs=-1)
evaluate("Random Forest + SMOTE", m3, X_smote, y_smote, aug_rows)

df_aug = pd.DataFrame(aug_rows)

# Find best baseline model for comparison
best_baseline = df_before.loc[df_before.F1.idxmax()]

print("\n" + "="*65)
print("AUGMENTATION RESULTS vs BASELINE (similar to Step 3 results)")
print("="*65)
print(f"\nBest baseline model: {best_baseline['Model']} — F1={best_baseline['F1']:.4f}")
print(f"\nAugmented results:")
print(df_aug[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].to_string(index=False))

# Comparison table: baseline vs augmented
print("\n" + "="*65)
print("AUGMENTATION COMPARISON: Baseline vs Augmented")
print("="*65)
comp_aug = pd.concat([
    df_before[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].assign(Type="Baseline (Before Tuning)"),
    df_aug[["Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].assign(Type="Augmented")
], ignore_index=True)
print(comp_aug.to_string(index=False))

df_aug.to_csv("results_augmented.csv", index=False)
print("\nSaved: results_augmented.csv")

# Augmentation bar chart
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle("Data Augmentation — Impact on Performance (vs Baseline)",
             fontsize=13, fontweight="bold")
aug_models = df_aug.Model.tolist()
colors_aug = ["#2196F3","#FF9800","#4CAF50"]
for ax, metric in zip(axes, ["Accuracy","Precision","Recall","F1"]):
    bars = ax.bar(range(len(aug_models)), df_aug[metric],
                  color=colors_aug, edgecolor="white", width=0.6)
    for bar, val in zip(bars, df_aug[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)
    ax.axhline(best_baseline[metric], color="red", ls="--", alpha=0.7,
               label=f"Baseline: {best_baseline[metric]:.4f}")
    ax.set_xticks(range(len(aug_models)))
    ax.set_xticklabels([m.replace(" + ","+\n") for m in aug_models], fontsize=8)
    ax.set_title(metric, fontweight="bold"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("augmentation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: augmentation_comparison.png")

## Cell 11 — Normalized Confusion Matrices (All Tuned Models)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 11))
fig.suptitle("Normalized Confusion Matrices — All Tuned Models (TF-IDF Text Features)",
             fontsize=14, fontweight="bold")

models_cm = [
    ("DT (Tuned)",    dt_final),
    ("LR (Tuned)",    lr_final),
    ("SVM (Tuned)",   svm_final),
    ("RF (Tuned)",    rf_final),
    ("XGB (Tuned)",   xgb_final),
    ("LGB (Tuned)",   lgb_final),
    ("Stacking MUST", stack_tuned),
]

for ax, (name, mdl) in zip(axes.flat, models_cm):
    pred = mdl.predict(X_test_dense)
    cm   = confusion_matrix(y_test_arr, pred, normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=["Control","Case"]).plot(
        ax=ax, colorbar=True, cmap="Blues", values_format=".2f")
    ax.set_title(name, fontsize=11, fontweight="bold")

# Hide last empty subplot
axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig("confusion_matrices_all.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices_all.png")

## Cell 12 — Learning Curves (Best Model)

In [ ]:
tr_sz, tr_sc, vl_sc = learning_curve(
    lgb_final, X_train_dense, y_train_arr,
    cv=3, scoring="f1_weighted",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Learning Curves — Best Model (LightGBM Tuned) on TF-IDF Text Features",
             fontsize=13, fontweight="bold")

axes[0].plot(tr_sz, tr_sc.mean(1), "o-", color="#2196F3", label="Train F1")
axes[0].fill_between(tr_sz, tr_sc.mean(1)-tr_sc.std(1), tr_sc.mean(1)+tr_sc.std(1), alpha=.15, color="#2196F3")
axes[0].plot(tr_sz, vl_sc.mean(1), "s-", color="#F44336", label="Val F1")
axes[0].fill_between(tr_sz, vl_sc.mean(1)-vl_sc.std(1), vl_sc.mean(1)+vl_sc.std(1), alpha=.15, color="#F44336")
axes[0].set_xlabel("Training Size"); axes[0].set_ylabel("F1 Score")
axes[0].set_title("Train vs Val F1"); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(tr_sz, 1-tr_sc.mean(1), "o-", color="#4CAF50", label="Train Error")
axes[1].plot(tr_sz, 1-vl_sc.mean(1), "s-", color="#FF9800", label="Val Error")
axes[1].set_xlabel("Training Size"); axes[1].set_ylabel("Error (1-F1)")
axes[1].set_title("Train vs Val Error"); axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150)
plt.show()
print("Saved: learning_curves.png")

## Cell 13 — XAI: SHAP Analysis (Best Model on Text Features)

In [ ]:
print("Computing SHAP values on TF-IDF text features...")

# Use 1000 test samples for speed
X_shap   = X_test_dense[:1000]
explainer = shap.TreeExplainer(lgb_final)
shap_vals = explainer.shap_values(X_shap)
sv = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

# Get TF-IDF feature names for the selected features
selected_feature_indices = selector.get_support(indices=True)
feature_names_all = tfidf.get_feature_names_out()
selected_feature_names = feature_names_all[selected_feature_indices]

import pandas as pd
X_shap_df = pd.DataFrame(X_shap, columns=selected_feature_names)

# Bar plot
plt.figure(figsize=(12, 8))
shap.summary_plot(sv, X_shap_df, plot_type="bar", max_display=20, show=False)
plt.title("SHAP Feature Importance — Top 20 TF-IDF Text Features (LightGBM Best)",
          fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("shap_bar.png", dpi=150)
plt.show()

# Beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(sv, X_shap_df, max_display=20, show=False)
plt.title("SHAP Beeswarm — Direction of Feature Effects", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=150)
plt.show()

# Save importance table
shap_imp = pd.DataFrame({
    "Feature":    selected_feature_names,
    "SHAP_Importance": np.abs(sv).mean(0)
}).sort_values("SHAP_Importance", ascending=False)
shap_imp.to_csv("shap_feature_importance.csv", index=False)
print("\nTop 10 most important TEXT features (by SHAP):")
print(shap_imp.head(10).to_string(index=False))
print("\nSaved: shap_bar.png, shap_beeswarm.png, shap_feature_importance.csv")

## Cell 14 — Ablation Study (Best Model on Text Features)

In [ ]:
print("Ablation Study — removing top SHAP text features incrementally")
top_features = shap_imp["Feature"].tolist()
ablation_results = []

# Map feature names back to column indices in X_train_dense
feat_to_idx = {name: i for i, name in enumerate(selected_feature_names)}

for k in [10, 20, 50, 100, 200, 500, 1000, 2000]:
    feats_k = [f for f in top_features[:k] if f in feat_to_idx]
    idx_k   = [feat_to_idx[f] for f in feats_k]
    if not idx_k: continue
    Xtr_k = X_train_dense[:, idx_k]
    Xte_k = X_test_dense[:, idx_k]
    m = LGBMClassifier(**{k_:v_ for k_,v_ in lgb_bayes.best_params_.items()},
                       random_state=42, verbosity=-1)
    m.fit(Xtr_k, y_train_arr)
    pred = m.predict(Xte_k)
    proba = m.predict_proba(Xte_k)[:,1]
    ablation_results.append({
        "Num_Features": len(idx_k),
        "Accuracy"    : round(accuracy_score(y_test_arr, pred), 4),
        "F1"          : round(f1_score(y_test_arr, pred, average="weighted"), 4),
        "AUC"         : round(roc_auc_score(y_test_arr, proba), 4),
    })
    print(f"  Top {len(idx_k):4d} features  F1={ablation_results[-1]['F1']:.4f}  AUC={ablation_results[-1]['AUC']:.4f}")

abl_df = pd.DataFrame(ablation_results)
print("\nAblation Table:")
print(abl_df.to_string(index=False))
abl_df.to_csv("ablation_study.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Ablation Study — Text Feature Count vs Performance", fontsize=13, fontweight="bold")
for ax, metric in zip(axes, ["Accuracy","F1","AUC"]):
    ax.plot(abl_df.Num_Features, abl_df[metric], "o-", color="#673AB7", lw=2, ms=7)
    bk = abl_df.loc[abl_df[metric].idxmax(), "Num_Features"]
    ax.axvline(bk, color="red", ls="--", alpha=.6, label=f"Best={bk}")
    ax.set_xlabel("# Text Features"); ax.set_ylabel(metric)
    ax.set_title(metric); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig("ablation_study.png", dpi=150)
plt.show()
print("Saved: ablation_study.png, ablation_study.csv")

## ✅ Notebook 1 Complete!

**Pipeline used:** Tabular CSV → Clinical Text Narratives → TF-IDF → Preprocessing → ML Models

**Files saved:**
- `results_before_tuning.csv` — 9 models before tuning
- `results_gridsearch.csv` — GridSearchCV all 6 models
- `results_randomsearch.csv` — RandomizedSearchCV all 6 models
- `results_bayesian.csv` — BayesSearchCV all 6 models
- `results_after_tuning.csv` — Best method per model + Stacking MUST
- `results_tuning_comparison.csv` — All 3 methods side-by-side
- `results_augmented.csv` — SMOTE + SMOTETomek results
- `confusion_matrices_all.png`
- `model_comparison_before_after.png`
- `tuning_methods_comparison.png`
- `learning_curves.png`
- `shap_bar.png` / `shap_beeswarm.png` / `shap_feature_importance.csv`
- `ablation_study.png` / `ablation_study.csv`

**→ Now run Notebook 2 on Colab with GPU (T4)**